In [ ]:
import os
import json
import json


# 2. Helper to get model prediction
def get_prediction(res):
    return res.get("prediction", res.get("predicted_answer"))


# 6. Compute per-concept precision, recall, F1
def compute_prf(metrics):
    prf = {}
    for c, m in metrics.items():
        tp, fp, fn = m["TP"], m["FP"], m["FN"]
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        prf[c] = {"precision": p, "recall": r, "f1": f1, "TP": tp, "FP": fp, "FN": fn}
    return prf


# 7. Compute micro and macro averages
def micro_avg(metrics):
    TP = sum(m["TP"] for m in metrics.values())
    FP = sum(m["FP"] for m in metrics.values())
    FN = sum(m["FN"] for m in metrics.values())
    p = TP / (TP + FP) if TP + FP else 0.0
    r = TP / (TP + FN) if TP + FN else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1

def macro_avg(prf):
    ps = [v["precision"] for v in prf.values()]
    rs = [v["recall"]    for v in prf.values()]
    f1s = [v["f1"]       for v in prf.values()]
    p = sum(ps) / len(ps) if ps else 0.0
    r = sum(rs) / len(rs) if rs else 0.0
    f1 = sum(f1s) / len(f1s) if f1s else 0.0
    return p, r, f1



## looping over files in "./gqa_atomic_benchmark/results/gqa_benchmark_test_*_concept_results.json"

for file in sorted(os.listdir("./gqa_atomic_benchmark/results/")):
    if file.endswith("_results.json"):
        import re
        method = re.search(r'(.*?)_results.json', file)
        print("="*20, method.group(1) if method else "Unknown", "="*20)
        print(f"Processing file: {file}")
        filename = os.path.join("./gqa_atomic_benchmark/results/", file)
        with open(filename, "r") as f:
            results = json.load(f)
        print(f"Loaded {len(results)} results from file {file}")
    else:
        continue

    # 3. Gather all concepts
    all_concepts = set(c for r in results for c in r.get("concepts", []))
    # filter behind, left, right, front, back, top, bottom
    for x in {"behind", "left", "right", "front", "back", "top", "bottom", "on", "in"}:
        all_concepts = {c for c in all_concepts if x not in c}
    # all_concepts = {c for c in all_concepts if any([x in c for x in ["behind", "left", "right", "front", "back", "top", "bottom", "on", "in"]])}
    
    print(f"Found {len(all_concepts)} unique concepts in the results.")
    # 4. Initialize metrics containers
    overall_metrics  = {c: {"TP": 0, "FP": 0, "FN": 0, "TN":0} for c in all_concepts}
    concept_metrics = {c: {"TP": 0, "FP": 0, "FN": 0, "TN":0} for c in all_concepts}
    property_metrics = {c: {"TP": 0, "FP": 0, "FN": 0, "TN":0} for c in all_concepts}
    relation_metrics = {c: {"TP": 0, "FP": 0, "FN": 0, "TN":0} for c in all_concepts}

    # 5. Tally TP, FP, FN, TN for each concept
    for r in results:
        ans = r.get("answer", "").strip()
        pred = get_prediction(r)
        is_pos_ans = ("Yes" in ans)
        is_pos_pred = (pred == "Yes")
        for c in r.get("concepts", []):
            if is_pos_pred and is_pos_ans:
                if c in overall_metrics:
                    overall_metrics[c]["TP"] += 1
            elif is_pos_pred and not is_pos_ans:
                if c in overall_metrics:
                    overall_metrics[c]["FP"] += 1
            elif not is_pos_pred and is_pos_ans:
                if c in overall_metrics:
                    overall_metrics[c]["FN"] += 1
            else:
                if c in overall_metrics:
                    overall_metrics[c]["TN"] += 1
            # relation-specific
            if r.get("type2") == "relation":
                if is_pos_pred and is_pos_ans:
                    if c in relation_metrics:
                        relation_metrics[c]["TP"] += 1
                elif is_pos_pred and not is_pos_ans:
                    if c in relation_metrics:
                        relation_metrics[c]["FP"] += 1
                elif not is_pos_pred and is_pos_ans:
                    if c in relation_metrics:
                        relation_metrics[c]["FN"] += 1
                else:
                    if c in relation_metrics:
                        relation_metrics[c]["TN"] += 1
            elif r.get("type2") == "property":
                if is_pos_pred and is_pos_ans:
                    if c in property_metrics:
                        property_metrics[c]["TP"] += 1
                elif is_pos_pred and not is_pos_ans:
                    if c in property_metrics:
                        property_metrics[c]["FP"] += 1
                elif not is_pos_pred and is_pos_ans:
                    if c in property_metrics:
                        property_metrics[c]["FN"] += 1
                else:
                    if c in property_metrics:
                        property_metrics[c]["TN"] += 1
            elif r.get("type2") == "class":
                if is_pos_pred and is_pos_ans:
                    if c in property_metrics:
                        concept_metrics[c]["TP"] += 1
                elif is_pos_pred and not is_pos_ans:
                    if c in property_metrics:
                        concept_metrics[c]["FP"] += 1
                elif not is_pos_pred and is_pos_ans:
                    if c in concept_metrics:
                        concept_metrics[c]["FN"] += 1
                else:
                    if c in property_metrics:
                        concept_metrics[c]["TN"] += 1
                # concept-specific    
    overall_prf  = compute_prf(overall_metrics)
    # only keep concepts that appear in relation set
    relation_prf_keys = [c for c in all_concepts if sum(relation_metrics[c].values()) > 0]
    concept_prf_keys = [c for c in all_concepts if sum(concept_metrics[c].values()) > 0]
    property_prf_keys = [c for c in all_concepts if sum(property_metrics[c].values()) > 0]
    # 6. Compute per-concept precision, recall, F1 

    relation_prf = compute_prf({c: relation_metrics[c] for c in relation_prf_keys})
    concept_prf  = compute_prf({c: concept_metrics[c] for c in concept_prf_keys})
    property_prf = compute_prf({c: property_metrics[c] for c in property_prf_keys})
    # 6. Compute overall metrics

    overall_micro  = micro_avg(overall_metrics)
    overall_macro  = macro_avg(overall_prf)
    
    concept_micro  = micro_avg(concept_metrics)
    concept_macro  = macro_avg(concept_prf)
    relation_micro = micro_avg({c: relation_metrics[c] for c in relation_prf_keys})
    relation_macro = macro_avg(relation_prf)
    property_micro = micro_avg({c: property_metrics[c] for c in property_prf_keys})
    property_macro = macro_avg(property_prf)


    # 8. Display the report
    ## Print overall TP, FP, FN 
    print(f"\n=== Overall Sum of Results for {method.group(1) if method else 'Unknown Method'} ===")
    print(f"{'Concept':30s} {'TP':>3s} {'FP':>3s} {'FN':>3s} {'TN':>3s}")
    total_tp = total_fp = total_fn = total_tn =  0
    for c, v in sorted(overall_metrics.items(), key=lambda x: -x[1]["TP"]):
        tp, fp, fn, tn = v["TP"], v["FP"], v["FN"], v["TN"]
        total_tp += tp; total_fp += fp; total_fn += fn; total_tn += tn
        # print(f"{c:30s} {tp:3d} {fp:3d} {fn:3d} {tn:3d}")
    print(f"{'Total':30s} {total_tp:3d} {total_fp:3d} {total_fn:3d} {total_tn:3d}")
    print("=== Overal Metrics ===")
    print(f"\nOverall Micro-Average: Precision={overall_micro[0]:.2%}, Recall={overall_micro[1]:.2%}, F1={overall_micro[2]:.2%}")
    print(f"Overall Macro-Average: Precision={overall_macro[0]:.2%}, Recall={overall_macro[1]:.2%}, F1={overall_macro[2]:.2%}\n")
    print("=== Concept Metrics ===")
    print(f"\nConcept Micro-Average: Precision={concept_micro[0]:.2%}, Recall={concept_micro[1]:.2%}, F1={concept_micro[2]:.2%}")
    print(f"Concept Macro-Average: Precision={concept_macro[0]:.2%}, Recall={concept_macro[1]:.2%}, F1={concept_macro[2]:.2%}")
    print("=== Relation Metrics ===")
    print(f"\nRelation Micro-Average: Precision={relation_micro[0]:.2%}, Recall={relation_micro[1]:.2%}, F1={relation_micro[2]:.2%}")
    print(f"Relation Macro-Average: Precision={relation_macro[0]:.2%}, Recall={relation_macro[1]:.2%}, F1={relation_macro[2]:.2%}")
    print("=== Property Metrics ===")
    print(f"\nProperty Micro-Average: Precision={property_micro[0]:.2%}, Recall={property_micro[1]:.2%}, F1={property_micro[2]:.2%}")
    print(f"Property Macro-Average: Precision={property_macro[0]:.2%}, Recall={property_macro[1]:.2%}, F1={property_macro[2]:.2%}")
    print("-" * 60)
    print()
    print()

    print("=== Overall Concept Metrics ===")
    print(f"{'Concept':30s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}       TP  FP  FN")
    for c, v in sorted(overall_prf.items(), key=lambda x: -x[1]["f1"]):
        print(f"{c:30s} {v['precision']:6.2%} {v['recall']:6.2%} {v['f1']:6.2%}   {v['TP']:3d} {v['FP']:3d} {v['FN']:3d}")

    print("=== Relation Concept Metrics ===")
    print(f"{'Concept':30s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}       TP  FP  FN")
    for c, v in sorted(relation_prf.items(), key=lambda x: -x[1]["f1"]):
        print(f"{c:30s} {v['precision']:6.2%} {v['recall']:6.2%} {v['f1']:6.2%}   {v['TP']:3d} {v['FP']:3d} {v['FN']:3d}")



In [ ]:
import json
from collections import Counter
from typing import List, Dict, Any

def _calculate_f1(data: List[Dict[str, Any]], concept: str) -> float:
    """
    Helper function to calculate the F1-score for a single concept in a given dataset.

    Args:
        data (List[Dict[str, Any]]): The evaluation data for a single model.
        concept (str): The concept to calculate the F1-score for.

    Returns:
        float: The calculated F1-score.
    """
    tp, tn, fp, fn = 0, 0, 0, 0

    for item in data:
        # Check if the current item is relevant to the concept
        concepts_in_item = []
        if item.get("subjectClassName"):
            concepts_in_item.append(item["subjectClassName"])
        if item.get("concepts"):
            concepts_in_item.extend(item["concepts"])

        if concept in concepts_in_item:
            predicted = str(item.get("predicted_answer", "")).strip().lower()
            ground_truth = str(item.get("short_answer", "")).strip().lower()

            if predicted == "yes" and ground_truth == "yes":
                tp += 1
            elif predicted == "no" and ground_truth == "no":
                tn += 1
            elif predicted == "yes" and ground_truth == "no":
                fp += 1
            elif predicted == "no" and ground_truth == "yes":
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return f1_score


def compare_models_f1(model_files: Dict[str, str]):
    """
    Loads data for multiple models, identifies the top 20 common concepts from the
    first file, and provides a side-by-side F1-score comparison.

    Args:
        model_files (Dict[str, str]): A dictionary mapping model names to their
                                     respective data file paths.
    """
    model_data = {}
    for model_name, file_path in model_files.items():
        try:
            with open(file_path, 'r') as f:
                model_data[model_name] = json.load(f)
        except FileNotFoundError:
            print(f"Error: The file '{file_path}' for model '{model_name}' was not found.")
            return
        except json.JSONDecodeError:
            print(f"Error: The file '{file_path}' is not a valid JSON file.")
            return

    if not model_data:
        print("No data was loaded.")
        return

    # --- 1. Identify top 20 concepts based on the FIRST file provided ---
    first_model_name = list(model_data.keys())[0]
    first_model_data = model_data[first_model_name]

    concept_counts = Counter()
    for item in first_model_data:
        concepts_in_item = []
        if item.get("subjectClassName"):
            concepts_in_item.append(item["subjectClassName"])
        if item.get("concepts"):
            concepts_in_item.extend(item["concepts"])
        for concept in set(concepts_in_item):
            concept_counts[concept] += 1
            
    top_20_concepts = concept_counts.most_common(20)

    # --- 2. Calculate F1-scores for each model and format results ---
    comparison_results = []
    for concept, frequency in top_20_concepts:
        row = {"Concept": concept, "Frequency": frequency}
        for model_name, data in model_data.items():
            f1_score = _calculate_f1(data, concept)
            row[f"{model_name} F1"] = f1_score
        comparison_results.append(row)

    # --- 3. Print the formatted comparison table ---
    model_names = list(model_data.keys())
    header = f"{'Concept':<15} | {'Frequency':<10} | " + " | ".join([f'{name[:12]:<12}' for name in model_names])
    
    print("\n--- Top 20 Common Concepts F1-Score Comparison ---")
    print(header)
    print("-" * len(header))

    for res in comparison_results:
        f1_scores_str = " | ".join([f"{res[f'{name} F1']:.3f}{'':<7}" for name in model_names])
        print(f"{res['Concept']:<15} | {res['Frequency']:<10} | {f1_scores_str}")
    print("-" * len(header))


# --- Example Usage ---
# To use this function, create three JSON files (or more) and then call the function
# with a dictionary mapping your chosen model names to the file paths.

if __name__ == '__main__':
    # Define the models and their corresponding data files
    files_to_compare = {
        "Ovis1.6-9B": "./gqa_atomic_benchmark/results/gqa_benchmark_templated_test_ovis_concept_results.json",
        "QwenVL2-7B": "./gqa_atomic_benchmark/results/gqa_benchmark_templated_test_qwen2_concept_results.json",
        "InternVL2.5-8B": "./gqa_atomic_benchmark/results/gqa_benchmark_templated_test_intern25_8b_concept_results.json",
        "VLLM": "./gqa_atomic_benchmark/results/gqa_benchmark_templated_test_intern_vllm_concept_results.json"
    }
    
    # Generate the comparison table
    compare_models_f1(files_to_compare)
    
    # This example call is commented out. To run it, you must:
    # 1. Create the JSON files with your data.
    # 2. Update the file paths in the `files_to_compare` dictionary.
    # 3. Uncomment the `compare_models_f1(files_to_compare)` line.
    print("Example call structure is ready. Please update file paths and uncomment the function call to run.")
# if __name__ == "__main__":

